# Step 2 — Model Training

Loads the preprocessed `dataset.npz` produced by `1_preprocess.ipynb` and trains a **ConvLSTM-based** binary classifier to distinguish violence from non-violence video sequences.

## Experiments performed

| # | Dataset | Architecture | Optimizer | LR | Batch | Accuracy |
|---|---------|-------------|-----------|-----|-------|----------|
| 1 | Original (1996) | 1× ConvLSTM | SGD | 1e-3 | 8 | 88% |
| 2 | Augmented (3992) | 1× ConvLSTM | SGD | 1e-3 | 8 | 91% |
| 3 | Augmented (3992) | 2× ConvLSTM (stacked) | SGD | 1e-3 | 8 | **93%** |

Run the cells top-to-bottom to reproduce the best result (Experiment 3).
Earlier experiments are preserved in separate sections for reference.

In [ ]:
import os
import random
import numpy as np
import keras
import tensorflow as tf
from keras.models import Sequential
from keras.layers import ConvLSTM2D, Dropout, Flatten, Dense
from keras.callbacks import EarlyStopping
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report

In [ ]:
# ── Reproducibility ───────────────────────────────────────────────────────────
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)
os.environ["TF_DETERMINISTIC_OPS"] = "1"

In [ ]:
# ── Configuration ─────────────────────────────────────────────────────────────
IMG_H   = 64
IMG_W   = 64
SEQ_LEN = 60
CLASSES = ["NonViolence", "Violence"]

In [ ]:
# ── Load preprocessed dataset ─────────────────────────────────────────────────
# Upload dataset.npz to Kaggle as a dataset, then update the path below.
data = np.load("/kaggle/input/video-classification-numpy-arrays/dataset.npz")
xx   = data["X"]   # shape: (N, 60, 64, 64, 3)
yy   = data["Y"]   # shape: (N, 2)  one-hot
print(f"X: {xx.shape}  |  Y: {yy.shape}")

---
## Helper: temporal augmentation

Doubles the dataset by reversing the frame order of each video.
This teaches the model temporal patterns in both directions without
collecting additional data.

In [ ]:
def augment_with_reverse(X, Y):
    """
    Double the dataset by appending temporally-reversed copies of each video.

    Args:
        X : (N, T, H, W, C)  video array
        Y : (N, num_classes) label array

    Returns:
        X_aug : (2N, T, H, W, C)
        Y_aug : (2N, num_classes)
    """
    X_reversed = X[:, ::-1, :, :, :]          # reverse along the time axis
    X_aug      = np.concatenate([X, X_reversed], axis=0)
    Y_aug      = np.concatenate([Y, Y],          axis=0)
    return X_aug, Y_aug

---
## Helper: model builders

In [ ]:
def build_convlstm_1layer(seq_len=SEQ_LEN, img_h=IMG_H, img_w=IMG_W, num_classes=2):
    """
    Baseline model: single ConvLSTM2D layer.

    Architecture:
        Input (seq_len, 64, 64, 3)
        → ConvLSTM2D(64, 3×3)
        → Dropout(0.2)
        → Flatten
        → Dense(256, ReLU) → Dropout(0.3)
        → Dense(2, Softmax)
    """
    model = Sequential([
        ConvLSTM2D(
            filters=64, kernel_size=(3, 3),
            return_sequences=False,
            data_format="channels_last",
            input_shape=(seq_len, img_h, img_w, 3)
        ),
        Dropout(0.2),
        Flatten(),
        Dense(256, activation="relu"),
        Dropout(0.3),
        Dense(num_classes, activation="softmax"),
    ])
    return model


def build_convlstm_2layer(seq_len=SEQ_LEN, img_h=IMG_H, img_w=IMG_W, num_classes=2):
    """
    Improved model: two stacked ConvLSTM2D layers.

    Architecture:
        Input (seq_len, 64, 64, 3)
        → ConvLSTM2D(64, 3×3, return_sequences=True)
        → Dropout(0.2)
        → ConvLSTM2D(64, 3×3, return_sequences=False)
        → Dropout(0.2)
        → Flatten
        → Dense(256, ReLU) → Dropout(0.3)
        → Dense(2, Softmax)
    """
    model = Sequential([
        ConvLSTM2D(
            filters=64, kernel_size=(3, 3),
            return_sequences=True,
            data_format="channels_last",
            input_shape=(seq_len, img_h, img_w, 3)
        ),
        Dropout(0.2),
        ConvLSTM2D(filters=64, kernel_size=(3, 3), return_sequences=False),
        Dropout(0.2),
        Flatten(),
        Dense(256, activation="relu"),
        Dropout(0.3),
        Dense(num_classes, activation="softmax"),
    ])
    return model

---
## Experiment 1 — Original dataset, 1-layer ConvLSTM, SGD lr=1e-3
Best result on original data: **88% accuracy**

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(xx, yy, test_size=0.20, shuffle=True, random_state=0)

model = build_convlstm_1layer()
model.compile(
    loss="categorical_crossentropy",
    optimizer=keras.optimizers.SGD(learning_rate=1e-3),
    metrics=["accuracy"]
)

history = model.fit(
    X_train, y_train,
    epochs=40, batch_size=8, shuffle=True, validation_split=0.2,
    callbacks=[EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)]
)
model.save("model_exp1.keras")

y_pred = np.argmax(model.predict(X_test), axis=1)
y_true = np.argmax(y_test, axis=1)
print(classification_report(y_true, y_pred, target_names=CLASSES))

---
## Experiment 2 — Augmented dataset, 1-layer ConvLSTM, SGD lr=1e-3
Dataset doubled via temporal reversal: **91% accuracy**

In [ ]:
X_aug, Y_aug = augment_with_reverse(xx, yy)
print(f"Augmented — X: {X_aug.shape}  Y: {Y_aug.shape}")

X_train, X_test, y_train, y_test = train_test_split(X_aug, Y_aug, test_size=0.20, shuffle=True, random_state=0)

model = build_convlstm_1layer()
model.compile(
    loss="categorical_crossentropy",
    optimizer=keras.optimizers.SGD(learning_rate=1e-3),
    metrics=["accuracy"]
)

history = model.fit(
    X_train, y_train,
    epochs=40, batch_size=8, shuffle=True, validation_split=0.2,
    callbacks=[EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)]
)
model.save("model_exp2.keras")

y_pred = np.argmax(model.predict(X_test), axis=1)
y_true = np.argmax(y_test, axis=1)
print(classification_report(y_true, y_pred, target_names=CLASSES))

---
## Experiment 3 — Augmented dataset, 2-layer ConvLSTM, SGD lr=1e-3  ✅ Best
Stacked ConvLSTM on augmented data: **93% accuracy** (final model)

In [ ]:
X_aug, Y_aug = augment_with_reverse(xx, yy)
X_train, X_test, y_train, y_test = train_test_split(X_aug, Y_aug, test_size=0.20, shuffle=True, random_state=0)

model = build_convlstm_2layer()
model.summary()
model.compile(
    loss="categorical_crossentropy",
    optimizer=keras.optimizers.SGD(learning_rate=1e-3),
    metrics=["accuracy"]
)

history = model.fit(
    X_train, y_train,
    epochs=40, batch_size=8, shuffle=True, validation_split=0.2,
    callbacks=[EarlyStopping(monitor="val_loss", patience=5, restore_best_weights=True)]
)
model.save("model.keras")   # final model used by the Flask app

y_pred = np.argmax(model.predict(X_test), axis=1)
y_true = np.argmax(y_test, axis=1)
print(classification_report(y_true, y_pred, target_names=CLASSES))